In [ ]:
from pathlib import Path
import pandas as pd

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import Structure
from qdk_chemistry.utils import Logger, compute_valence_space_parameters
from qdk_chemistry.utils.wavefunction import (
    calculate_sparse_wavefunction,
    get_active_determinants_info,
)

In [ ]:
structure_path = Path.cwd() / "data" / "ozone.structure.xyz"
structure = Structure.from_xyz_file(structure_path)
Logger.info(structure.get_summary())

scf_solver = create("scf_solver")
e_scf, scf_wavefunction = scf_solver.run(structure, 0, 1, "cc-pvdz")
Logger.info(f"SCF energy = {e_scf:.8f} Hartree")

electrons, orbitals = compute_valence_space_parameters(scf_wavefunction, 0)
selector = create("active_space_selector", "qdk_valence")
settings = selector.settings()
settings.set("num_active_electrons", electrons)
settings.set("num_active_orbitals", orbitals)

active_orbital_wavefunction = selector.run(scf_wavefunction)
active_orbitals = active_orbital_wavefunction.get_orbitals()
Logger.info(active_orbitals.get_summary())

hamiltonian_constructor = create("hamiltonian_constructor")
active_hamiltonian = hamiltonian_constructor.run(active_orbitals)
Logger.info(active_hamiltonian.get_summary())

casci_calculator = create(
    "multi_configuration_calculator", "macis_asci"
)
casci_calculator.settings().set("calculate_one_rdm", True)
casci_calculator.settings().set("calculate_two_rdm", True)
casci_calculator.settings().set("core_selection_strategy", "fixed")
e_cas, wfn_cas = casci_calculator.run(
     active_hamiltonian, *active_orbital_wavefunction.get_active_num_electrons()
)
Logger.info(f"CASCI energy = {e_cas:.8f} Hartree")

autocas_selector = create("active_space_selector", "qdk_autocas")
refined_wfn = autocas_selector.run(wfn_cas)
indices, _ = refined_wfn.get_orbitals().get_active_space_indices()
Logger.info(f"autoCAS selected active space with indices: {indices}")

refined_orbitals = refined_wfn.get_orbitals()
active_hamiltonian = hamiltonian_constructor.run(refined_orbitals)
e_cas, wfn_cas = casci_calculator.run(
    active_hamiltonian, *refined_wfn.get_active_num_electrons()
)
Logger.info(active_hamiltonian.get_summary())
Logger.info(f"autoCAS energy = {e_cas:.8f} Hartree")

sparse_ci_energy, sparse_ci_wavefunction = calculate_sparse_wavefunction(
    reference_wavefunction=wfn_cas,
    hamiltonian=active_hamiltonian,
    reference_energy=e_cas,
    energy_tolerance=1.0e-3,
    max_determinants=100,
)

Logger.info(f"Sparse CI energy = {sparse_ci_energy:.8f} Hartree")
Logger.info(get_active_determinants_info(sparse_ci_wavefunction))

In [ ]:
from qdk.widgets import Circuit

sparse_isometry_gf2x = create("state_prep", "sparse_isometry_gf2x")

gf2x_circuit = sparse_isometry_gf2x.run(sparse_ci_wavefunction)
display(Circuit(gf2x_circuit.get_qsharp_circuit()))

# Print logical qubit counts estimated from the circuit
df = pd.DataFrame(
    gf2x_circuit.estimate().logical_counts.items(), columns=['Logical Estimate', 'Counts'])
display(df)

In [ ]:
from qdk.widgets import Circuit

sparse_isometry_bin_enc = create("state_prep", "sparse_isometry_binary_encoding", measurement_based_uncompute=False)

bin_enc_circuit = sparse_isometry_bin_enc.run(sparse_ci_wavefunction)
#display(Circuit(bin_enc_circuit.get_qsharp_circuit()))

# Print logical qubit counts estimated from the circuit
df = pd.DataFrame(
    bin_enc_circuit.estimate().logical_counts.items(), columns=['Logical Estimate', 'Counts'])
display(df)

In [ ]:
from utils.state_prep_utils import filter_and_group_pauli_ops_from_wavefunction
from qdk_chemistry.data import QubitHamiltonian

qubit_mapper = create("qubit_mapper", "qdk")
qubit_hamiltonian = qubit_mapper.run(active_hamiltonian)

pauli_strings =[]
coeffs = []
filtered_ops, classical_coeffs = filter_and_group_pauli_ops_from_wavefunction(qubit_hamiltonian, sparse_ci_wavefunction)
for op, _ in zip(filtered_ops, classical_coeffs):
    pauli_strings.extend(op.pauli_strings)
    coeffs.extend(op.coefficients)
filtered_hamiltonian = QubitHamiltonian(pauli_strings, coeffs)

In [ ]:
import numpy as np

energy_estimator = create("energy_estimator", "qdk")
simulator = create("circuit_executor", "qiskit_aer_simulator")

results, _ = energy_estimator.run(bin_enc_circuit, filtered_hamiltonian, simulator, 240000)

# Print statistic for measured energy
energy_mean = results.energy_expectation_value + active_hamiltonian.get_core_energy() + sum(classical_coeffs)
energy_stddev = np.sqrt(results.energy_variance)
print(
    f"Estimated energy from quantum circuit: {energy_mean:.3f} ± {energy_stddev:.3f} Hartree"
)

# Print comparison with reference energy
print(f"Difference from reference energy: {energy_mean - sparse_ci_energy} Hartree")